# 11b: Autocorrelation, Stationarity, and Differencing

On Tuesday, we separated trend-cycle and seasonality from monthly nuclear generation. The remainder contained what those estimated components did not explain. We ended with a question: does a positive remainder tend to be followed by another positive remainder?

Today we will **describe relationships between observations separated in time**, then **ask whether those relationships remain consistent** through the record. We will examine both levels and changes.

The methods used herein can be applied to an original series, a seasonally adjusted series, or a decomposition remainder; always identify which one is being examined.

## Sources

Adapted from *Forecasting: Principles and Practice, the Pythonic Way* by Rob J Hyndman, George Athanasopoulos, Azul Garza, Cristian Challu, Max Mergenthaler, and Kin G Olivares (OTexts, 2026): [§§2.7–2.9](https://otexts.com/fpppy/02-graphics.html#lag-plots) and selected parts of [§9.1](https://otexts.com/fpppy/09-arima.html#stationarity-and-differencing).

Data sources and extraction details are in [data/README.md](data/README.md).

## Setup

Load the three datasets and check their time indexes. The table reports the number of observations and the interval represented by each series.

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import acf

# Locate the three local extracts before using the public source files.
DATA_URL = "https://otexts.com/fpppy/data/"
data_candidates = [
    Path("data"),
    Path("lectures/11b-autocorrelation-and-stationarity/data"),
    Path("content/lectures/11b-autocorrelation-and-stationarity/data"),
]
data_names = ["aus-beer.csv", "google-2015.csv", "a10-monthly.csv"]
data_dir = next((p for p in data_candidates
                 if all((p / name).exists() for name in data_names)), None)


# Read a public CSV when the local extracts are unavailable.
def upstream_csv(name):
    request = Request(DATA_URL + name, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request, timeout=30) as response:
        return pd.read_csv(response)


if data_dir is not None:
    beer_frame = pd.read_csv(data_dir / "aus-beer.csv")
    google_frame = pd.read_csv(data_dir / "google-2015.csv")
    a10_frame = pd.read_csv(data_dir / "a10-monthly.csv")
else:
    beer_frame = upstream_csv("aus_production.csv")[["ds", "Beer"]]
    stocks = upstream_csv("gafa_stock.csv")
    stock_dates = pd.to_datetime(stocks["ds"])
    google_frame = stocks.loc[
        (stocks["unique_id"] == "GOOG_Close") & (stock_dates.dt.year == 2015),
        ["ds", "y"],
    ].rename(columns={"y": "Close"})
    pbs = upstream_csv("PBS_unparsed.csv")
    a10_frame = (pbs.loc[pbs["ATC2"] == "A10"].groupby("Month")["Cost"]
                 .sum().div(1e6).rename_axis("ds").reset_index())


# Parse actual date labels and return one checked, time-indexed series.
def indexed(frame, value):
    frame = frame.copy()
    frame["ds"] = pd.to_datetime(frame["ds"], format="%Y-%m-%d")
    result = frame.set_index("ds")[value]
    assert result.index.is_unique and result.index.is_monotonic_increasing
    assert result.notna().all()
    return result


# Match the lecture's dates and check the complete quarterly/monthly calendars.
beer = indexed(beer_frame, "Beer").loc["2000":]
google = indexed(google_frame, "Close")
a10 = indexed(a10_frame, "Cost")
assert beer.index.equals(pd.date_range(beer.index[0], beer.index[-1], freq="QS"))
assert a10.index.equals(pd.date_range(a10.index[0], a10.index[-1], freq="MS"))
assert (a10 > 0).all()

plt.rcParams.update({"figure.dpi": 110, "font.size": 12,
                     "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE = "#185c8c", "#c45116"

display(pd.DataFrame({"Series": ["Beer", "Google", "A10"],
                      "Observations": [len(beer), len(google), len(a10)],
                      "Interval": ["Quarter", "Trading observation", "Month"]}))

`indexed` converts the date column into a time index and selects one measurement column. The assertions check ordering, unique dates, available values, and complete quarterly/monthly calendars. The Google series retains its observed trading dates.

## Lag

A **lag** is the number of periods back we look. At time $t$, the current observation is $y_t$ and the observation at lag $h$ is $y_{t-h}$. For quarterly data, lag 1 means the previous quarter and lag 4 means the same quarter a year earlier.

We will use the beer-production series from Tuesday, selecting 2000 onward. Each value is Australian production during one quarter, in megalitres. The date labels the start of the quarter.

In [ ]:
# Display the selected quarters on their original production scale.
fig, ax = plt.subplots(figsize=(10, 3.5), layout="constrained")
ax.plot(beer.index, beer, "o-", color=BLUE, markersize=4)
ax.set(title="Quarterly Australian beer production", xlabel="Year", ylabel="Megalitres")
plt.show()
plt.close(fig)

Before plotting pairs, inspect their alignment. Each row below attaches the previous quarter’s value and the previous year’s corresponding quarter to the current observation.

In [ ]:
# Align earlier production with each current observation's date.
beer_pairs = pd.DataFrame({
    "Current": beer,
    "Lag 1": beer.shift(1),
    "Lag 4": beer.shift(4),
})
display(beer_pairs.head(8))

`shift(1)` places the previous row’s value beside the current date; `shift(4)` does the same for four rows earlier. The initial `NaN` entries mark dates without an earlier partner in the selected record.

For 2001 Q1, the current value is 451 ML, lag 1 is 500 ML from 2000 Q4, and lag 4 is 421 ML from 2000 Q1. The first four rows have no lag-4 value within the selected record. Those pairs are unavailable because the earlier quarters fall outside the selected record.

The quarterly calendar is complete, so four rows back really is four quarters back. With missing periods, a row shift could pair observations separated by a different amount of time.

### Lag Plots

A lag plot shows how an observed value at time ($y_t$) relates to the lagged value $h$ periods prior ($y_{t-h}$). Each point in a lag plot represents one $y_{t-h}$, $y_t$ pair. The horizontal coordinate is the earlier value; the vertical coordinate is the current value.

The following panels vary the beer production lag from one through nine quarters. Colors identify the quarter of the current observation, and every panel uses the same axes.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(10, 9), sharex=True, sharey=True,
                         layout="constrained")
quarter_colors = ["#185c8c", "#c45116", "#16745b", "#934a91"]
limits = (0.95 * beer.min(), 1.05 * beer.max())
# Build a separate set of current/earlier pairs for each lag.
for h, ax in enumerate(axes.flat, start=1):
    pairs = pd.DataFrame({"Current": beer, "Earlier": beer.shift(h)}).dropna()
    # Color each point by the quarter of the current observation.
    for quarter, color in enumerate(quarter_colors, start=1):
        selected = pairs.loc[pairs.index.quarter == quarter]
        ax.scatter(selected["Earlier"], selected["Current"], color=color,
                   label=f"Q{quarter}", s=25)
    ax.plot(limits, limits, "--", color="0.7", lw=0.8)
    ax.set(title=f"Lag {h}", xlim=limits, ylim=limits, aspect="equal")
handles, labels = axes[0, 2].get_legend_handles_labels()
fig.legend(handles, labels, title="Current quarter", loc="outside upper center", ncol=4)
fig.supxlabel("Earlier production (ML)")
fig.supylabel("Current production (ML)")
plt.show()
plt.close(fig)

For each lag, `.dropna()` removes pairs with an unavailable earlier value. `pairs.index.quarter` identifies the current quarter for the point colors. Shared axis limits make the nine relationships directly comparable.

Which lag values, $h$, show the greatest correlation between observed values $y_t$ and $y_{t-h}$?

Lags 4 and 8 pair corresponding quarters across years: high values tend to pair with high values and low with low. Lags 2 and 6 often pair opposite parts of the annual cycle, producing a negative relationship. The lag changes which relationship we examine.

Now let’s quantify the relationship at each lag.

## Autocorrelation

Correlation measures the strength and direction of a linear relationship between two variables.

$$
r_{xy}=\frac{\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i-\bar{x})^2}\sqrt{\sum_{i=1}^{n}(y_i-\bar{y})^2}}
$$

Values near +1 indicate a strong positive linear relationship, values near −1 a strong negative linear relationship, and values near zero little linear relationship.

**Autocorrelation** measures the linear relationship between values in a series and its own earlier values. There is a separate coefficient for each lag. A positive value indicates that above-average observations tend to pair with above-average observations; a negative value indicates that above-average observations tend to pair with below-average observations.

For $T$ observations with overall mean $\bar y$, the sample autocorrelation at lag $h$ is

$$
r_h = \frac{\sum_{t=h+1}^{T}(y_t-\bar y)(y_{t-h}-\bar y)}{\sum_{t=1}^{T}(y_t-\bar y)^2}
$$

The numerator combines the centered products for available pairs. Products are positive when the two values are on the same side of the mean and negative when they are on opposite sides. The denominator puts the result on a common scale using the variation in the full series.

Calculate lag 4 manually using this sample-ACF estimator. It uses one overall mean and denominator across lags. Pearson correlation instead centers and scales each pair of shortened columns separately, so the two calculations can give slightly different results.

In [ ]:
h = 4
# Center once using the mean of the full selected series.
centered = beer - beer.mean()
# Only complete current/lagged pairs contribute to the numerator.
products = (centered * centered.shift(h)).dropna()
numerator = products.sum()
# Use all centered observations for the common normalization.
denominator = (centered ** 2).sum()
r4 = numerator / denominator
display(pd.DataFrame({"Centered current": centered,
                      "Centered lag 4": centered.shift(h),
                      "Product": centered * centered.shift(h)}).dropna().head())
print(f"Lag 4: {numerator:.2f} / {denominator:.2f} = {r4:.3f}")

Multiplication aligns the two Series by their date labels. After shifting, `.dropna()` keeps the complete pairs for the numerator. The denominator sums squared deviations over the full series, matching the displayed formula. The resulting lag-4 coefficient is about 0.802.

The **autocorrelation function**, or ACF, collects the correlations across lag values. The `acf` function estimates them from our series. Display lags 1–9 to connect them with the nine lag plots.

In [ ]:
# Estimate autocorrelations at lags 0 through 16.
beer_acf = acf(beer, nlags=16, fft=False, adjusted=False)
assert np.isclose(r4, beer_acf[4])

# Display lags 1 through 9; array position zero is lag zero.
display(pd.Series(beer_acf[1:10], index=pd.Index(range(1, 10), name="Lag"),
                  name="Autocorrelation").round(3).to_frame().T)

`nlags=16` requests estimates through lag 16; the returned array contains 17 values because it includes lag zero. `adjusted=False` uses the common full-series normalization in our formula. `fft=False` computes the sums directly. `[1:10]` selects positions 1 through 9, and `np.isclose` checks agreement with our manual lag-4 calculation.

The lag-4 coefficient is about 0.802, while lag 2 is about −0.758. The signs and magnitudes summarize the relationships visible in their scatterplots.

### ACF Plots

An ACF plot shows each coefficient as a vertical spike. Lag zero is omitted because comparing the series with itself without shifting gives a correlation of one. We will explain the shaded reference band in the next section.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5), layout="constrained")
# Omit lag zero and use the white-noise reference band.
plot_acf(beer, lags=16, zero=False, bartlett_confint=False, ax=ax)
ax.set(title="Beer production: autocorrelation", xlabel="Lag (quarters)", ylabel="ACF")
plt.show()
plt.close(fig)

`zero=False` leaves the self-correlation at lag zero off the chart. `bartlett_confint=False` selects the white-noise reference band; its meaning is developed in the next section.

### Trend and Seasonality in the ACF

A repeating seasonal pattern can produce positive correlations at multiples of its period. A changing level can also produce strong autocorrelation: observations close in time share a similar level, even when they come from different parts of a long upward trend.

Australian antidiabetic drug sales illustrate this combination. Here `Cost` is the monthly total cost of A10 prescriptions, expressed in millions of Australian dollars, from July 1991 through June 2008. The following helper displays a time plot beside its ACF. Use it to compare the sales pattern with the correlations across lags.

In [ ]:
def show_series_acf(series, title, ylabel, lag_unit, lags=24):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), layout="constrained")
    # Left panel: values in their original time order.
    axes[0].plot(series.index, series, color=BLUE, lw=1.2)
    axes[0].set(title=title, ylabel=ylabel,
                xlabel="Date" if isinstance(series.index, pd.DatetimeIndex) else "Period")
    # Right panel: correlations across lags, with white-noise reference bounds.
    plot_acf(series, lags=lags, zero=False, bartlett_confint=False,
             ax=axes[1], auto_ylims=True)
    # Keep the correlation axis within its possible range.
    low, high = axes[1].get_ylim()
    axes[1].set_ylim(max(-1, low), min(1, high))
    axes[1].set(title="Autocorrelation", xlabel=f"Lag ({lag_unit})", ylabel="ACF")
    plt.show()
    plt.close(fig)

show_series_acf(a10, "Antidiabetic drug sales", "AUD millions", "months", lags=48)

`show_series_acf` takes a Series, a title, measurement units, lag units, and the largest lag to display. It creates a time plot and an ACF using the same observations. Later calls reuse these display settings with different series.

The series rises over time and has larger annual swings at higher levels. Its ACF combines persistent positive correlations with peaks at annual lags. This raises another question: does the relationship stay consistent through the record?

## White Noise

What would the ACF look like without underlying time dependence? How might we use that to create a numerical test?

A **white-noise process** has mean zero, constant finite variance, and zero autocorrelation at every nonzero lag. Independent random draws with the same distribution provide one way to generate it. The ACF measures linear dependence. Other forms of dependence can exist even when the autocorrelations are zero.

Generate 50 independent standard-normal values and examine their time plot and ACF. The generating process has no autocorrelation; the estimated correlations in this one short sample will still fluctuate.

In [ ]:
# Fix the seed so this illustration is reproducible.
random = np.random.RandomState(30)
# Draw 50 independent values with mean zero and standard deviation one.
white_noise = pd.Series(random.normal(0, 1, 50), index=np.arange(1, 51))
show_series_acf(white_noise, "Simulated white noise", "Value", "periods", lags=16)

`normal(0, 1, 50)` draws 50 values with mean zero and standard deviation one. The seed fixes the sample for repeatable discussion. The index labels the generated observations 1 through 50.

We can use white noise as a reference: how large might a sample autocorrelation be when the underlying autocorrelation is zero?

It can be shown that, for a sufficiently long white-noise series, the sample autocorrelation at a specified nonzero lag falls approximately 95% of the time between

$$
-\frac{1.96}{\sqrt{T}} \quad\text{and}\quad +\frac{1.96}{\sqrt{T}}
$$

Here $T$ is the number of observations. For 50 observations, the bounds are approximately ±0.28. An estimated autocorrelation inside this band is compatible with the random variation expected around zero; one outside it gives evidence of linear dependence at that lag. A longer record gives narrower bounds.

These are **pointwise** bounds: the 95% statement applies to one specified lag. When we inspect many lags, chance crossings become more likely. Our simulated sample illustrates this: its first spike is just outside the band even though the observations were generated independently.

## Stationarity

Autocorrelation summarizes pairs taken from different parts of the record. When is it reasonable to describe those pairs with one relationship? We need to ask whether the properties of the process stay consistent through time.

A **stationary** time series is one whose statistical properties do not depend on the time at which the series is observed. More precisely, if $\{y_t\}$ is a stationary time series, then for all $s$, the distribution of $(y_t,\ldots,y_{t+s})$ does not depend on $t$.

A changing mean, a changing spread, or a fixed calendar-seasonal pattern violates stationarity. Our independent standard-normal draws have the same distribution at every time and are stationary. A stationary process can also have dependence, provided its joint distributions remain unchanged when shifted in time.

We will compare Google’s 2015 closing prices with their changes. First, inspect the levels alongside our other examples. Each Google row is an observed trading day. Lag 1 means the previous trading observation; a weekend or exchange holiday can separate consecutive observations by several calendar days.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), layout="constrained")
examples = [(google, "Google closing price, 2015", "USD"),
            (beer, "Quarterly beer production", "ML"),
            (a10, "Monthly antidiabetic drug sales", "AUD millions"),
            (white_noise, "Simulated white noise", "Value")]
# Keep each series in its own units while comparing its time pattern.
for ax, (series, title, units) in zip(axes.flat, examples):
    ax.plot(series.index, series, color=BLUE, lw=1)
    ax.set(title=title, ylabel=units,
           xlabel="Date" if isinstance(series.index, pd.DatetimeIndex) else "Period")
plt.show()
plt.close(fig)

Google's level changes substantially during the year. Beer has a recurring quarterly pattern. Drug sales combine growth with increasing seasonal swings. Those patterns challenge different stationarity conditions. The simulated noise fluctuates around a common level with no built-in change in its spread.

Assess whether the level, spread, and temporal relationships remain consistent across different parts of each record.

## Ordinary Differencing

Instead of examining the level, we can examine how much it changed since the previous observation. The **first difference** is

$$
\Delta y_t = y_t-y_{t-1}
$$

The new quantity is a change per observation interval. For Google's prices it is the dollar change since the preceding trading observation.

Show the original price, earlier price, and their difference before examining the complete series. The price labels below keep the actual trading dates.

In [ ]:
# Subtract the previous trading observation; remove the initial unavailable difference.
google_change = google.diff().dropna()
display(pd.DataFrame({"Closing price": google,
                      "Previous close": google.shift(1),
                      "Change": google.diff()}).head(6).round(3))
assert len(google_change) == len(google) - 1

`.diff()` subtracts the previous row’s value. The first difference is unavailable and `.dropna()` removes it, leaving 251 changes. The retained dates identify the trading observation at which each change ends.

Differencing changes the analytical question from “How high is the price?” to “How much did it move?” A changing level can become a series of changes with more consistent behavior.

Compare the levels and changes, each with its ACF.

In [ ]:
# Compare levels with changes using the same lag range.
show_series_acf(google, "Google closing price, 2015", "USD", "trading observations")
show_series_acf(google_change, "Change since previous close", "USD change",
                "trading observations")

The level has a persistent ACF. The changes fluctuate around a much more stable level and their ACF is much quieter, although a large individual price move remains. Interpret the ACF together with the time plot: persistent correlations can occur with a changing level or with stationary dependence.

### A Joint Check for Autocorrelation

The **Ljung–Box test** checks autocorrelations jointly over a selected range of lags. Its null hypothesis here is that the population autocorrelations at lags 1 through 10 are all zero. A small p-value provides evidence against that hypothesis. A large p-value means the observed autocorrelations are compatible with that hypothesis.

Apply the test to the price changes over the first ten lags.

In [ ]:
# Jointly test zero autocorrelation at lags 1 through 10.
ljung_box = acorr_ljungbox(google_change, lags=[10], return_df=True)
display(ljung_box.round(3))

`lags=[10]` requests one joint test covering lags 1–10. The output row is labeled by that maximum lag. `lb_stat` is the test statistic and `lb_pvalue` is its p-value; the following interpretation uses the p-value.

The p-value is about 0.637. The autocorrelations across the first ten lags are compatible with the white-noise reference, agreeing with the relatively quiet ACF.

[Further reading: tests for autocorrelation, FPPpy §5.4](https://otexts.com/fpppy/05-toolbox.html#portmanteau-tests-for-autocorrelation).

## Random Walks

How can a series have a wandering level while its changes have no autocorrelation? Consider a process that starts at a fixed value and adds a new, independent, zero-mean random change at each step:

$$
y_t = y_{t-1}+\varepsilon_t
$$

This is a **random walk**. Here $\varepsilon_t$ is the new random change. Rearranging gives

$$
y_t-y_{t-1}=\varepsilon_t
$$

The levels accumulate the changes; differencing recovers them. A run of positive or negative changes can produce apparent trends without a persistent upward or downward force.

Simulate 200 independent changes and accumulate them, starting at zero. Compare the wandering path with the changes that generated it.

In [ ]:
rng = np.random.default_rng(7130)
# Generate independent changes, then accumulate them into levels.
steps = rng.normal(0, 1, 200)
# Prepend the fixed starting level so all 200 changes can be recovered.
walk = pd.Series(np.r_[0.0, np.cumsum(steps)])
walk_change = walk.diff().dropna()
# Check that differencing recovers the changes used to construct the walk.
np.testing.assert_allclose(walk_change, steps)
show_series_acf(walk, "Simulated random walk", "Level", "periods")
show_series_acf(walk_change, "Random-walk changes", "Change", "periods")

`np.cumsum(steps)` accumulates the random changes. `np.r_[0.0, ...]` places the starting level before those accumulated values, giving 201 levels for 200 changes. Applying `.diff()` recovers the increments, which the assertion verifies.

The changes were generated independently, but neighboring levels share most of the same accumulated history. That creates strong dependence in the levels.

This random walk is not stationary: after more steps, more independent changes have accumulated, increasing the uncertainty about its level. With step variance one and a fixed starting value, the variance of the level after $t$ steps is $t$. A constant expected level alone is not sufficient for stationarity.

If future changes have expected value zero, the current level is the expected future level. That gives a simple prediction: use the latest observed value. The next observation will also include a new random change.

The quiet ACF of Google’s price changes illustrates why this simple prediction is worth evaluating.

## Seasonal Differencing

For a seasonal series, consecutive observations can belong to very different parts of the cycle. We can instead compare corresponding positions in successive cycles. With $m$ observations per cycle, the **seasonal difference** is

$$
\Delta_m y_t = y_t-y_{t-m}
$$

For monthly observations with an annual pattern, $m=12$. The result measures the change since the same month last year. We lose the first 12 observations, which have no year-earlier counterpart in the record.

This differs from Tuesday's seasonal adjustment. Seasonal adjustment subtracts an estimated seasonal component and retains the level. Seasonal differencing subtracts an earlier observation and produces a change.

### Same-Month Comparisons

Return to the antidiabetic drug sales. First calculate year-over-year changes on the original scale so their meaning is visible. Each subtraction compares two monthly totals in millions of Australian dollars.

In [ ]:
# Compare each monthly total with the same month one year earlier.
a10_annual_change = a10.diff(12)
display(pd.DataFrame({"Current sales": a10,
                      "Same month last year": a10.shift(12),
                      "Annual change": a10_annual_change}).iloc[12:18].round(3))
assert a10_annual_change.notna().sum() == len(a10) - 12

`.diff(12)` subtracts the value 12 rows earlier. The checked monthly calendar makes this a same-month-last-year comparison. `.iloc[12:18]` displays the first six available annual changes, beginning in July 1992.

We also saw that the seasonal swings grow with the level. Recall the log transformation: equal proportional changes become equal differences on the log scale. Because these sales are positive, we can take their logs and then compare the same month across years.

In [ ]:
# Stabilize proportional swings, then calculate year-over-year log changes.
log_a10 = np.log(a10)
seasonal_change = log_a10.diff(12).dropna()

`np.log` transforms each positive monthly total. `.diff(12)` then compares those logs one year apart; `.dropna()` removes the initial 12 unavailable comparisons, leaving 192 annual log changes.

The result has a useful interpretation:

$$
\log(y_t)-\log(y_{t-12}) = \log\left(\frac{y_t}{y_{t-12}}\right)
$$

A value of zero means no year-over-year change; a positive value means growth. A log difference of 0.10 corresponds to a ratio of $e^{0.10}$, or about a 10.5% increase. Small log differences approximate proportional changes.

Plot the sales, their logs, and the seasonal differences of the logs. The panels use different units, so compare their patterns rather than their vertical magnitudes.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True, layout="constrained")
# Share dates across panels; label each panel in its own units.
for ax, series, label in zip(
    axes, [a10, log_a10, seasonal_change],
    ["Sales (AUD millions)", "Log sales", "Annual log change"],
):
    ax.plot(series.index, series, color=BLUE, lw=1.2)
    ax.set_ylabel(label)
axes[0].set_title("Antidiabetic drug sales: logs and seasonal differences")
axes[2].axhline(0, color="0.6", lw=0.8)
axes[-1].set_xlabel("Year")
plt.show()
plt.close(fig)

The log scale makes the annual fluctuations more comparable across the record. Seasonal differencing then removes much of that repeating pattern and the sustained rise in the level. The resulting series is easier to assess as fluctuations around a reasonably consistent average annual change.

Compare the ACF before and after seasonal differencing. Does the transformed series still contain dependence?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), layout="constrained")
# Apply the same ACF settings before and after seasonal differencing.
for ax, series, title in zip(
    axes, [log_a10, seasonal_change], ["Log sales", "Seasonal differences of log sales"]
):
    plot_acf(series, lags=36, zero=False, bartlett_confint=False,
             ax=ax, auto_ylims=True)
    ax.set(title=title, xlabel="Lag (months)", ylabel="ACF")
plt.show()
plt.close(fig)

The persistent pattern changes substantially, but correlations remain. A stationary series can retain dependence that a forecasting model can use. Choose any further differencing to address a remaining change in level or seasonal behavior, and assess its effect.

If the same-month change had expected value zero and no useful predictable structure, the corresponding simple prediction would be last year's value for that month. We will evaluate predictions such as these against later observations in the forecasting lecture.

### Choosing the Difference

The subtraction should match an interpretable question: change since the previous period, or change since the previous corresponding season. Afterward, inspect both the time plot and the ACF. Unnecessary differencing can introduce dependence and make the series harder to model.

*Optional note:* A second ordinary difference is a change in consecutive first differences. It is not a lag-2 difference. A seasonal difference can also be followed by an ordinary difference when justified; that is a different operation from two ordinary differences.

## What the Evidence Supports

We began with the question left by decomposition: is there still a relationship between observations separated in time?

- Lag plots make the pairs visible; autocorrelation summarizes their linear relationships.
- White noise supplies a reference for the sample variation we can see even without underlying autocorrelation.
- Stationarity asks whether statistical behavior remains consistent when we shift our observation window through time.
- Ordinary and seasonal differences change the quantity under study, which can make its behavior more consistent.

The next question is whether any remaining structure helps us predict observations that were not available when we made the prediction. That requires a forecast and an honest comparison with what happened afterward.